# 🔍 VerifAI — Multimodal Fake News Detection System
### Deep Learning Project

---

This notebook demonstrates the complete **multimodal fake news detection pipeline** using:
- 📝 **Text Analysis** — BERT-based transformer + linguistic feature extraction
- 🖼️ **Image Analysis** — Error Level Analysis (ELA) + metadata forensics
- 🎥 **Video Analysis** — Frame sampling + temporal consistency
- 🔀 **Multimodal Fusion** — Confidence-weighted combination

---

## 📦 Cell 1 — Install Required Packages

In [ ]:
import subprocess, sys

packages = [
    'flask', 'torch', 'transformers', 'Pillow',
    'opencv-python', 'numpy', 'scipy', 'werkzeug',
    'accelerate', 'sentencepiece', 'matplotlib',
    'seaborn', 'ipywidgets'
]

for pkg in packages:
    print(f'Installing {pkg}...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f'  [OK] {pkg}')
    else:
        print(f'  [FAIL] {pkg}: {result.stderr[:100]}')

print('\n[DONE] All packages installed!')

## 🛠️ Cell 2 — Setup & Imports

In [ ]:
import os
import sys
import io
import re
import json
import warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image, ImageChops

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# Add project root to path so we can import our models
PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print('[OK] Imports successful')
print('Project root:', PROJECT_ROOT)

## 🎨 Cell 3 — Visualization Helper

In [ ]:
def plot_verdict(result, title='Analysis Result'):
    """Plot the verdict and probability bars for any modality result."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#0a0a1f')

    fake_p = result.get('fake_probability', 0.5)
    real_p = result.get('real_probability', 0.5)
    verdict = result.get('verdict', 'UNCERTAIN')
    confidence = result.get('confidence', 0.5)

    # --- Donut chart ---
    ax1 = axes[0]
    ax1.set_facecolor('#0a0a1f')
    colors = ['#ef4444', '#10b981']
    wedges, texts, autotexts = ax1.pie(
        [fake_p, real_p],
        labels=['FAKE', 'REAL'],
        colors=colors,
        autopct='%1.1f%%',
        startangle=90,
        wedgeprops=dict(width=0.55, edgecolor='#0a0a1f', linewidth=3),
        pctdistance=0.75
    )
    for text in texts + autotexts:
        text.set_color('white')
        text.set_fontsize(13)
    ax1.set_title(f'{title}\nVerdict: {verdict}  |  Confidence: {confidence*100:.1f}%',
                  color='white', fontsize=13, fontweight='bold', pad=20)

    # Center label
    color = '#ef4444' if verdict == 'FAKE' else '#10b981'
    ax1.text(0, 0, verdict, ha='center', va='center',
             fontsize=22, fontweight='black', color=color)

    # --- Bar chart for features (if text) ---
    ax2 = axes[1]
    ax2.set_facecolor('#0a0a1f')
    features = result.get('features', {})

    if features:
        score_keys = ['clickbait_score', 'emotional_score', 'caps_ratio',
                      'vague_source_score', 'credibility_score']
        labels_map = {
            'clickbait_score': 'Clickbait', 'emotional_score': 'Emotional',
            'caps_ratio': 'CAPS Ratio', 'vague_source_score': 'Vague Sources',
            'credibility_score': 'Credibility'
        }
        vals = [features.get(k, 0) for k in score_keys]
        labels_list = [labels_map[k] for k in score_keys]
        bar_colors = ['#ef4444' if v > 0.3 else '#7c3aed' for v in vals]
        bar_colors[-1] = '#10b981'  # credibility is always green

        bars = ax2.barh(labels_list, vals, color=bar_colors,
                        edgecolor='none', height=0.6)
        ax2.set_xlim(0, 1)
        ax2.set_xlabel('Score', color='white', fontsize=11)
        ax2.set_title('Linguistic Feature Scores', color='white',
                      fontsize=13, fontweight='bold')
        ax2.tick_params(colors='white', labelsize=11)
        ax2.spines[['top','right','bottom','left']].set_color('#334155')
        ax2.set_facecolor('#0d0d2b')
        for bar, val in zip(bars, vals):
            ax2.text(min(val + 0.02, 0.95), bar.get_y() + bar.get_height()/2,
                     f'{val:.2f}', va='center', color='white', fontsize=10)
    else:
        ela = result.get('ela_analysis', {})
        meta = result.get('metadata', {})
        visual = result.get('visual_analysis', {})
        comp = result.get('compression_analysis', {})

        metrics = {
            'ELA Score': ela.get('ela_score', 0),
            'No EXIF': 1.0 if not meta.get('has_exif') else 0.0,
            'Edit Software': 1.0 if meta.get('software_edit') else 0.0,
            'Color Inconsist.': visual.get('color_inconsistency', 0),
            'Compression': comp.get('artifact_score', 0),
        }
        bar_colors2 = ['#ef4444' if v > 0.3 else '#7c3aed' for v in metrics.values()]
        bars2 = ax2.barh(list(metrics.keys()), list(metrics.values()),
                         color=bar_colors2, edgecolor='none', height=0.6)
        ax2.set_xlim(0, 1)
        ax2.set_xlabel('Score', color='white', fontsize=11)
        ax2.set_title('Image Forensic Metrics', color='white',
                      fontsize=13, fontweight='bold')
        ax2.tick_params(colors='white', labelsize=11)
        ax2.spines[['top','right','bottom','left']].set_color('#334155')
        ax2.set_facecolor('#0d0d2b')
        for bar, val in zip(bars2, metrics.values()):
            ax2.text(min(val + 0.02, 0.95), bar.get_y() + bar.get_height()/2,
                     f'{val:.2f}', va='center', color='white', fontsize=10)

    plt.tight_layout(pad=2)
    plt.show()


def plot_fusion(fusion_result):
    """Plot the full multimodal fusion result."""
    mod_results = fusion_result.get('modality_results', {})
    modalities = list(mod_results.keys())

    if not modalities:
        print('No modality results to plot.')
        return

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.patch.set_facecolor('#0a0a1f')
    fig.suptitle('VerifAI — Multimodal Fusion Result', color='white',
                 fontsize=16, fontweight='black', y=1.02)

    # 1. Overall verdict donut
    ax1 = axes[0]
    ax1.set_facecolor('#0d0d2b')
    fp = fusion_result.get('fake_probability', 0.5)
    rp = fusion_result.get('real_probability', 0.5)
    wedges, _, autotexts = ax1.pie(
        [fp, rp], colors=['#ef4444', '#10b981'],
        autopct='%1.1f%%', startangle=90,
        wedgeprops=dict(width=0.6, edgecolor='#0a0a1f', linewidth=3),
        pctdistance=0.78
    )
    for at in autotexts:
        at.set_color('white'); at.set_fontsize(12)
    verdict = fusion_result.get('verdict', 'UNCERTAIN')
    vcolor = '#ef4444' if verdict == 'FAKE' else '#10b981'
    ax1.text(0, 0.05, verdict, ha='center', va='center',
             fontsize=24, fontweight='black', color=vcolor)
    risk = fusion_result.get('risk_level', '')
    ax1.text(0, -0.18, f'Risk: {risk}', ha='center', va='center',
             fontsize=12, color='#94a3b8')
    ax1.set_title('Final Verdict', color='white', fontsize=13, fontweight='bold')
    ax1.legend(['FAKE', 'REAL'], loc='lower center', ncol=2,
               labelcolor='white', facecolor='#0a0a1f', edgecolor='#334155',
               fontsize=10, bbox_to_anchor=(0.5, -0.08))

    # 2. Per-modality fake probability bars
    ax2 = axes[1]
    ax2.set_facecolor('#0d0d2b')
    icons = {'text': '[TXT]', 'image': '[IMG]', 'video': '[VID]'}
    mod_fps = [mod_results[m].get('fake_probability', 0.5) * 100 for m in modalities]
    mod_rps = [mod_results[m].get('real_probability', 0.5) * 100 for m in modalities]
    x = np.arange(len(modalities))
    w = 0.35
    b1 = ax2.bar(x - w/2, mod_fps, w, label='Fake %', color='#ef4444', alpha=0.85,
                 edgecolor='none')
    b2 = ax2.bar(x + w/2, mod_rps, w, label='Real %', color='#10b981', alpha=0.85,
                 edgecolor='none')
    ax2.set_xticks(x)
    ax2.set_xticklabels([m.capitalize() for m in modalities], color='white', fontsize=12)
    ax2.set_ylabel('Probability (%)', color='white', fontsize=11)
    ax2.set_ylim(0, 110)
    ax2.tick_params(colors='white')
    ax2.spines[['top','right']].set_visible(False)
    ax2.spines[['left','bottom']].set_color('#334155')
    ax2.legend(labelcolor='white', facecolor='#0a0a1f',
               edgecolor='#334155', fontsize=10)
    ax2.set_title('Per-Modality Scores', color='white', fontsize=13, fontweight='bold')
    for bar in list(b1) + list(b2):
        h = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2, h + 1.5,
                 f'{h:.1f}%', ha='center', color='white', fontsize=9)

    # 3. Confidence & weight table
    ax3 = axes[2]
    ax3.set_facecolor('#0d0d2b')
    ax3.axis('off')
    table_data = []
    cols = ['Modality', 'Verdict', 'Fake %', 'Conf %', 'Weight']
    for m in modalities:
        mr = mod_results[m]
        table_data.append([
            m.upper(),
            mr.get('verdict', '?'),
            '{:.1f}%'.format(mr.get('fake_probability', 0.5) * 100),
            '{:.1f}%'.format(mr.get('confidence', 0.5) * 100),
            '{:.2f}'.format(mr.get('base_weight', 0)),
        ])
    # Add fusion row
    table_data.append(['FUSION', verdict,
                       '{:.1f}%'.format(fp * 100),
                       '{:.1f}%'.format(fusion_result.get('confidence', 0) * 100),
                       '---'])
    tbl = ax3.table(cellText=table_data, colLabels=cols,
                    loc='center', cellLoc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(11)
    tbl.scale(1.2, 2.0)
    for (row, col), cell in tbl.get_celld().items():
        cell.set_facecolor('#1e1e3f' if row % 2 == 0 else '#0d0d2b')
        cell.set_text_props(color='white')
        cell.set_edgecolor('#334155')
        if row == 0:
            cell.set_facecolor('#7c3aed')
            cell.set_text_props(color='white', fontweight='bold')
        if row == len(table_data):  # fusion row
            fc = '#ef444433' if verdict == 'FAKE' else '#10b98133'
            cell.set_facecolor(fc)
    ax3.set_title('Modality Summary', color='white', fontsize=13,
                  fontweight='bold', pad=20)

    plt.tight_layout(pad=2)
    plt.show()

print('[OK] Visualization helpers defined')

## 📝 Cell 4 — Text Analysis
Paste any news article or claim below. The system will analyse it with BERT + linguistic features.

In [ ]:
from models.text_analyzer import TextAnalyzer

print('Loading Text Analyzer (BERT model may download on first run)...')
text_analyzer = TextAnalyzer()
print('[OK] Text Analyzer ready')

In [ ]:
# ============================================================
# CHANGE THIS TEXT to any news article or claim you want to test
# ============================================================
SAMPLE_FAKE = """
SHOCKING!! You WON'T BELIEVE what the government is hiding from you!!
EXCLUSIVE: Insiders reveal a massive conspiracy that mainstream media refuses to report.
Sources say that the entire economy is about to COLLAPSE and they are censoring this news.
Share this before it gets BANNED! This explosive bombshell will DESTROY everything you knew!
Allegedly, some experts claim this could be the end. People are saying this is devastating.
"""

SAMPLE_REAL = """
According to a study published in the journal Nature, scientists say that global temperatures
have risen by 1.1 degrees Celsius since pre-industrial times. Data shows that the last decade
was the hottest on record. The research, peer-reviewed by climate scientists, confirms that
greenhouse gas emissions are the primary driver. Official statements from the IPCC have
verified these findings, which are confirmed by multiple independent research institutions.
"""

# Analyse both
print('=' * 60)
print('EXAMPLE 1 — Likely FAKE text')
print('=' * 60)
result_fake_text = text_analyzer.analyze(SAMPLE_FAKE)
print(f"  Verdict      : {result_fake_text['verdict']}")
print(f"  Fake Prob    : {result_fake_text['fake_probability']*100:.1f}%")
print(f"  Real Prob    : {result_fake_text['real_probability']*100:.1f}%")
print(f"  Confidence   : {result_fake_text['confidence']*100:.1f}%")
print(f"  Model Used   : {result_fake_text.get('model_used', 'N/A')}")
print()
print('Features:')
for k, v in result_fake_text['features'].items():
    print(f"  {k:25s}: {v}")
print()
print('Indicators:')
for ind in result_fake_text.get('manipulation_indicators', []):
    print(f"  -> {ind}")

print()
print('=' * 60)
print('EXAMPLE 2 — Likely REAL text')
print('=' * 60)
result_real_text = text_analyzer.analyze(SAMPLE_REAL)
print(f"  Verdict      : {result_real_text['verdict']}")
print(f"  Fake Prob    : {result_real_text['fake_probability']*100:.1f}%")
print(f"  Real Prob    : {result_real_text['real_probability']*100:.1f}%")
print(f"  Confidence   : {result_real_text['confidence']*100:.1f}%")

In [ ]:
# Plot FAKE text result
plot_verdict(result_fake_text, title='Text Analysis — Fake News Sample')

In [ ]:
# Plot REAL text result
plot_verdict(result_real_text, title='Text Analysis — Real News Sample')

In [ ]:
# ============================================================
# ENTER YOUR OWN TEXT HERE
# ============================================================
MY_TEXT = """
Paste your own news article or suspicious text here and run this cell.
"""

if MY_TEXT.strip() and 'Paste your own' not in MY_TEXT:
    my_text_result = text_analyzer.analyze(MY_TEXT)
    print(f"Verdict: {my_text_result['verdict']}")
    print(f"Fake Probability: {my_text_result['fake_probability']*100:.1f}%")
    plot_verdict(my_text_result, title='Your Custom Text Analysis')
else:
    print('[INFO] Replace the text in MY_TEXT above and re-run this cell.')

## 🖼️ Cell 5 — Image Analysis (ELA Forensics)

In [ ]:
from models.image_analyzer import ImageAnalyzer

print('Loading Image Analyzer...')
image_analyzer = ImageAnalyzer()
print('[OK] Image Analyzer ready')

In [ ]:
import urllib.request
import os

# Download sample test images (public domain)
SAMPLE_IMAGE_URL = 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png'
SAMPLE_IMAGE_PATH = 'test_image.png'

if not os.path.exists(SAMPLE_IMAGE_PATH):
    print('Downloading sample image...')
    try:
        urllib.request.urlretrieve(SAMPLE_IMAGE_URL, SAMPLE_IMAGE_PATH)
        print('[OK] Sample image downloaded:', SAMPLE_IMAGE_PATH)
    except Exception as e:
        print('[WARN] Could not download:', e)
        print('Please place your own image and update SAMPLE_IMAGE_PATH below.')
else:
    print('[OK] Sample image already exists:', SAMPLE_IMAGE_PATH)

# ============================================================
# CHANGE THIS to the path of your image file
# Examples: 'news_image.jpg', r'C:\Users\..\photo.jpg'
# ============================================================
IMAGE_TO_ANALYZE = SAMPLE_IMAGE_PATH

In [ ]:
# Run image analysis
if os.path.exists(IMAGE_TO_ANALYZE):
    print('Analysing image:', IMAGE_TO_ANALYZE)
    image_result = image_analyzer.analyze(IMAGE_TO_ANALYZE)

    print(f"\nVerdict         : {image_result['verdict']}")
    print(f"Fake Probability: {image_result['fake_probability']*100:.1f}%")
    print(f"Real Probability: {image_result['real_probability']*100:.1f}%")
    print(f"Confidence      : {image_result['confidence']*100:.1f}%")
    print(f"Image Size      : {image_result.get('image_size', 'N/A')}")

    ela = image_result.get('ela_analysis', {})
    meta = image_result.get('metadata', {})
    print(f"\nELA Score       : {ela.get('ela_score', 0):.4f}  (higher = more suspicious)")
    print(f"Has EXIF        : {meta.get('has_exif', False)}")
    print(f"Edit Software   : {meta.get('software_edit', False)} ({meta.get('software', 'None')})")
    print(f"Faces Detected  : {image_result.get('face_detection', {}).get('face_count', 0)}")

    print('\nManipulation Indicators:')
    for ind in image_result.get('manipulation_indicators', ['None detected']):
        print(f'  -> {ind}')
else:
    print('[ERROR] Image not found:', IMAGE_TO_ANALYZE)

In [ ]:
# Visualize image + ELA map side by side
if os.path.exists(IMAGE_TO_ANALYZE):
    orig = Image.open(IMAGE_TO_ANALYZE).convert('RGB')

    # Compute ELA map
    buf = io.BytesIO()
    orig.save(buf, format='JPEG', quality=85)
    buf.seek(0)
    recomp = Image.open(buf).convert('RGB')
    ela_diff = ImageChops.difference(orig, recomp)
    ela_arr = np.array(ela_diff)
    ela_enhanced = (ela_arr * 10).clip(0, 255).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.patch.set_facecolor('#0a0a1f')

    # Original
    axes[0].imshow(orig)
    axes[0].set_title('Original Image', color='white', fontsize=13, fontweight='bold')
    axes[0].axis('off')

    # ELA map
    axes[1].imshow(ela_enhanced)
    axes[1].set_title(
        'ELA Map (Error Level Analysis)\nBrighter = more suspicious regions',
        color='white', fontsize=11, fontweight='bold'
    )
    axes[1].axis('off')

    # ELA heatmap (grayscale mean)
    ela_gray = ela_arr.mean(axis=2)
    im = axes[2].imshow(ela_gray, cmap='hot')
    axes[2].set_title('ELA Heatmap\n(Hot = High Manipulation Likelihood)',
                      color='white', fontsize=11, fontweight='bold')
    axes[2].axis('off')
    cbar = plt.colorbar(im, ax=axes[2], fraction=0.046)
    cbar.ax.tick_params(colors='white')

    plt.tight_layout()
    plt.show()

    # Also show verdict plot
    plot_verdict(image_result, title='Image Analysis Result')

## 🎥 Cell 6 — Video Analysis

In [ ]:
from models.video_analyzer import VideoAnalyzer

print('Loading Video Analyzer...')
video_analyzer = VideoAnalyzer()
print('[OK] Video Analyzer ready')

In [ ]:
# ============================================================
# CHANGE THIS to the path of your video file
# Examples: 'news_clip.mp4', r'C:\Users\..\video.mp4'
# ============================================================
VIDEO_TO_ANALYZE = 'your_video.mp4'

if os.path.exists(VIDEO_TO_ANALYZE):
    print('Analysing video:', VIDEO_TO_ANALYZE)
    video_result = video_analyzer.analyze(VIDEO_TO_ANALYZE)

    vi = video_result.get('video_info', {})
    ta = video_result.get('temporal_analysis', {})
    frames = video_result.get('frame_analysis', [])

    print(f"\nVerdict              : {video_result['verdict']}")
    print(f"Fake Probability     : {video_result['fake_probability']*100:.1f}%")
    print(f"Confidence           : {video_result['confidence']*100:.1f}%")
    print(f"Duration             : {vi.get('duration_sec', 0):.1f}s")
    print(f"Resolution           : {vi.get('resolution', 'N/A')}")
    print(f"FPS                  : {vi.get('fps', 0):.1f}")
    print(f"Frames Analysed      : {vi.get('analyzed_frames', 0)} / {vi.get('total_frames', 0)}")
    print(f"Temporal Consistency : {ta.get('consistency_score', 0)*100:.1f}%")
    print(f"Anomalous Frames     : {ta.get('anomalous_frames', 0)}")

    if frames:
        # Plot frame-by-frame fake probability
        timestamps = [f['timestamp'] for f in frames]
        fake_probs = [f['fake_probability'] * 100 for f in frames]
        real_probs = [(1 - f['fake_probability']) * 100 for f in frames]

        fig, ax = plt.subplots(figsize=(14, 5))
        fig.patch.set_facecolor('#0a0a1f')
        ax.set_facecolor('#0d0d2b')

        ax.fill_between(timestamps, fake_probs, alpha=0.3, color='#ef4444')
        ax.fill_between(timestamps, real_probs, alpha=0.15, color='#10b981')
        ax.plot(timestamps, fake_probs, 'o-', color='#ef4444', linewidth=2.5,
                markersize=8, label='Fake Probability (%)')
        ax.plot(timestamps, real_probs, 's--', color='#10b981', linewidth=2,
                markersize=6, label='Real Probability (%)')
        ax.axhline(50, color='#f59e0b', linestyle='--', linewidth=1.5,
                   alpha=0.7, label='Decision Threshold (50%)')

        ax.set_xlabel('Timestamp (seconds)', color='white', fontsize=12)
        ax.set_ylabel('Probability (%)', color='white', fontsize=12)
        ax.set_title('Frame-by-Frame Fake Probability Over Time',
                     color='white', fontsize=14, fontweight='bold')
        ax.set_ylim(0, 105)
        ax.tick_params(colors='white')
        ax.spines[['top','right']].set_visible(False)
        ax.spines[['left','bottom']].set_color('#334155')
        ax.legend(labelcolor='white', facecolor='#0a0a1f',
                  edgecolor='#334155', fontsize=11)
        plt.tight_layout()
        plt.show()
else:
    print(f'[INFO] No video found at: {VIDEO_TO_ANALYZE}')
    print('Change VIDEO_TO_ANALYZE to your video file path and re-run.')

    # Demo: simulate frame results
    print('\n[DEMO] Showing simulated video frame analysis...')
    np.random.seed(42)
    sim_times = np.linspace(0, 30, 12)
    sim_probs = np.clip(np.random.normal(0.65, 0.15, 12), 0.1, 0.95)

    fig, ax = plt.subplots(figsize=(14, 5))
    fig.patch.set_facecolor('#0a0a1f')
    ax.set_facecolor('#0d0d2b')
    ax.fill_between(sim_times, sim_probs * 100, alpha=0.25, color='#ef4444')
    ax.plot(sim_times, sim_probs * 100, 'o-', color='#ef4444',
            linewidth=2.5, markersize=8, label='Fake Probability (%)')
    ax.axhline(50, color='#f59e0b', linestyle='--', linewidth=1.5,
               alpha=0.7, label='Decision Threshold')
    ax.set_xlabel('Timestamp (seconds)', color='white', fontsize=12)
    ax.set_ylabel('Probability (%)', color='white', fontsize=12)
    ax.set_title('[DEMO] Simulated Frame-by-Frame Analysis',
                 color='white', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 105)
    ax.tick_params(colors='white')
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['left','bottom']].set_color('#334155')
    ax.legend(labelcolor='white', facecolor='#0a0a1f',
              edgecolor='#334155', fontsize=11)
    plt.tight_layout()
    plt.show()

## 🔀 Cell 7 — Multimodal Fusion

In [ ]:
from models.fusion import FusionModel

fusion = FusionModel()
print('[OK] Fusion model ready')

# Use whichever results we have
modality_inputs = {}

# Always include text result
modality_inputs['text'] = result_fake_text

# Include image result if available
if 'image_result' in dir():
    modality_inputs['image'] = image_result

# Include video result if available
if 'video_result' in dir() and os.path.exists(VIDEO_TO_ANALYZE):
    modality_inputs['video'] = video_result

print(f'Fusing {list(modality_inputs.keys())} modalities...')
fusion_result = fusion.combine(modality_inputs)

print()
print('=' * 60)
print('FINAL FUSION RESULT')
print('=' * 60)
print(f"  Verdict         : {fusion_result['verdict']}")
print(f"  Fake Probability: {fusion_result['fake_probability']*100:.1f}%")
print(f"  Real Probability: {fusion_result['real_probability']*100:.1f}%")
print(f"  Confidence      : {fusion_result['confidence']*100:.1f}%")
print(f"  Risk Level      : {fusion_result['risk_level']}")
print(f"  Modalities Used : {fusion_result['modalities_used']}")
print(f"  Fusion Method   : {fusion_result['fusion_method']}")
print()
print('Manipulation Indicators:')
for ind in fusion_result.get('manipulation_indicators', ['None']):
    print(f'  -> {ind}')

In [ ]:
# Full fusion visualization
plot_fusion(fusion_result)

## 📊 Cell 8 — Comparison: Fake vs Real

In [ ]:
# Compare FAKE vs REAL text side by side
res_fake = text_analyzer.analyze(SAMPLE_FAKE)
res_real = text_analyzer.analyze(SAMPLE_REAL)

feature_names = ['clickbait_score', 'emotional_score', 'caps_ratio',
                 'vague_source_score', 'credibility_score']
labels_nice = ['Clickbait', 'Emotional', 'CAPS Ratio', 'Vague Sources', 'Credibility']

fake_vals = [res_fake['features'].get(k, 0) for k in feature_names]
real_vals = [res_real['features'].get(k, 0) for k in feature_names]

x = np.arange(len(labels_nice))
w = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#0a0a1f')
ax.set_facecolor('#0d0d2b')

b1 = ax.bar(x - w/2, fake_vals, w, label='FAKE news sample',
            color='#ef4444', alpha=0.85, edgecolor='none')
b2 = ax.bar(x + w/2, real_vals, w, label='REAL news sample',
            color='#10b981', alpha=0.85, edgecolor='none')

ax.set_xticks(x)
ax.set_xticklabels(labels_nice, color='white', fontsize=12)
ax.set_ylabel('Feature Score (0 to 1)', color='white', fontsize=12)
ax.set_title('Fake vs Real — Linguistic Feature Comparison',
             color='white', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.tick_params(colors='white')
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#334155')
ax.legend(labelcolor='white', facecolor='#0a0a1f',
          edgecolor='#334155', fontsize=11)

for bar in list(b1) + list(b2):
    h = bar.get_height()
    if h > 0.01:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.02,
                f'{h:.2f}', ha='center', color='white', fontsize=9)

# Add fake probability comparison text
ax.text(0.02, 0.95,
        f"FAKE sample -> Fake Prob: {res_fake['fake_probability']*100:.1f}%",
        transform=ax.transAxes, color='#ef4444', fontsize=11, fontweight='bold')
ax.text(0.02, 0.89,
        f"REAL sample -> Fake Prob: {res_real['fake_probability']*100:.1f}%",
        transform=ax.transAxes, color='#10b981', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 🌐 Cell 9 — Launch Web App (Optional)
Run this cell to start the Flask web server for the full UI experience.

In [ ]:
import threading
import webbrowser
import time

def run_flask():
    import sys
    import io
    if sys.platform == 'win32':
        sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8', errors='replace')
    from app import app
    import os
    os.makedirs('static/uploads', exist_ok=True)
    app.run(debug=False, port=5000, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

time.sleep(2)
print('[OK] Flask server started!')
print('Opening browser at http://localhost:5000 ...')
webbrowser.open('http://localhost:5000')
print('[OK] Web app is running. Keep this cell running to maintain the server.')

---
## 📋 Summary

| Modality | Method | Key Signals |
|----------|--------|-------------|
| **Text** | BERT transformer + Linguistic rules | Clickbait, emotional language, CAPS, vague sources |
| **Image** | Error Level Analysis (ELA) + Metadata | JPEG inconsistency, missing EXIF, editing software |
| **Video** | Frame sampling + Temporal analysis | Per-frame ELA, inter-frame variance, anomaly detection |
| **Fusion** | Confidence-weighted average | Combined verdict with risk level |

---
*VerifAI — Deep Learning Project | PyTorch + HuggingFace Transformers + OpenCV + Flask*